# Setting up relative binding free energy networks from two protein conformations

This notebook starts matched RBFE campaigns from open (`3a7r`) and closed (`1x2h`) protein models. Both use the same chemical identities, charges, atom mappings, and an aligned canonical ligand pose set. This controls some setup differences, but shared poses still need inspection in both pockets; matching the alignment frame does not establish a correct binding mode.

**Interpretation:** these unrestrained runs start in different conformations; they do not by themselves define separate equilibrium ensembles. If both explore the same conformational ensemble and converge, their ΔΔG should agree. A reproducible difference can be interpreted as basin-conditioned only after confirming that all endpoints/repeats remain in consistently defined basins (or using a separately validated restraint scheme with its free-energy accounting).

With `δ_conf = ΔG_bind,conf(B) - ΔG_bind,conf(A)`, `δ_open - δ_closed` is the change in open-versus-closed preference from ligand A to B. It does not determine which conformation a single ligand prefers: that requires a reference preference or conformational free-energy offset. Population-weighted overall binding needs the corresponding apo populations. A shared solvent estimate cancels in the open-minus-closed edge difference; include covariance if reusing it in uncertainty propagation. Keep open/closed result trees separate when using `openfe gather`, since ligand-pair names repeat across conformations.


In [ ]:
import multiprocessing as mp
import os
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path

import MDAnalysis as mda
import mdtraj as md
import numpy as np
import openfe
from openfe import (
    AlchemicalNetwork,
    ChemicalSystem,
    ProteinComponent,
    SmallMoleculeComponent,
    SolventComponent,
    Transformation,
)
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol
from openfe.protocols.openmm_septop import SepTopProtocol
from openfe.protocols.openmm_utils.charge_generation import bulk_assign_partial_charges
from openfe.protocols.openmm_utils.omm_settings import OpenFFPartialChargeSettings
from openfe.setup import RBFEAlchemicalNetworkPlanner, RHFEAlchemicalNetworkPlanner
from openfe.setup.atom_mapping import KartografAtomMapper, LomapAtomMapper
from openfe.setup.atom_mapping.lomap_scorers import (
    atomic_number_score,
    default_lomap_score,
    ecr_score,
    mcsr_score,
    mncar_score,
)
from openfe.setup.ligand_network_planning import (
    generate_lomap_network,
    generate_maximal_network,
    generate_minimal_redundant_network,
    generate_minimal_spanning_network,
    generate_network_from_indices,
    generate_network_from_names,
    generate_radial_network,
)
from openfe.utils.atommapping_network_plotting import plot_atommapping_network
from openff.units import unit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Descriptors3D import Asphericity
from workflow_utils import input_digest

from mdpp.plots import draw_mols, make_atom_labels_3d, view_mol_3d, view_traj_3d
from mdpp.prep import assign_topology, fix_pdb

## Configuration

Define input paths, chain identifiers, and simulation parameters. The two protein conformations are declared in `CONFORMATIONS` (name -> PDB filename token), and `complex_pdb_for(name, conformation)` is the single place mapping a (ligand, conformation) pair to its complex PDB -- edit only it (and `CONFORMATIONS`) to change how inputs are resolved. `select_ligand_conformation` decides, per ligand, which conformation's pose becomes the single canonical ligand that is charged once and reused for both conformations. Proteins and OpenFE transformation outputs are written per conformation (`*_open`, `*_closed`); the ligand, charge, and network paths are shared.

In [ ]:
COMPLEX_DIR = Path("pdbs")
SMILES_PATH = Path("ligands_amp_fep.smi")
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"

# The two protein conformations to compare: conformation name -> PDB filename
# token. Both conformations have a modeled complex for every ligand.
CONFORMATIONS = {"open": "3a7r", "closed": "1x2h"}
# Conformation whose template protein defines the single alignment frame; the
# one canonical ligand pose set must sit in both pockets, so all complexes
# (open and closed) are superposed onto this conformation's reference.
ALIGN_REFERENCE_CONFORMATION = "open"
# Atoms used to seed the superposition of every complex onto the common frame
# (MDAnalysis selection). The open and closed states differ by a large
# C-terminal-domain reorientation (residues ~247-337 move up to ~44 A), so a
# single least-squares fit -- on the whole protein, or even all of the NTD --
# settles on a poor global-minimum compromise and misplaces the ligand. We fit
# the rigid N-terminal domain (residues 1-246) and refine it PyMOL-style with
# iterative outlier rejection (align_to_reference): atoms deviating more than
# ALIGN_CUTOFF are dropped and the fit repeated up to ALIGN_MAX_CYCLES times, so
# the flexible NTD loop (~166-182) is excluded automatically and the rigid core
# drives the alignment (NTD RMSD ~0.5 A vs ~3.3 A single-shot).
ALIGN_SELECTION = "protein and backbone and resid 1:246"
ALIGN_CUTOFF = 2.0  # Angstrom; reject atom pairs deviating more than this per cycle
ALIGN_MAX_CYCLES = 5  # max outlier-rejection refinement cycles


def complex_pdb_for(name: str, conformation: str) -> Path:
    """Resolve a (ligand name, conformation) pair to its complex PDB path.

    Ligand names from `SMILES_PATH` are the single source of truth and
    `conformation` is a key of `CONFORMATIONS`. This is the only place that
    encodes how a (ligand, conformation) pair maps to a complex PDB file. To
    change the mapping (a different filename template, a lookup table, more
    conformations, etc.), edit only this function and `CONFORMATIONS`.

    Args:
        name: Ligand name, i.e. the title field of an entry in `SMILES_PATH`.
        conformation: Key of `CONFORMATIONS` (e.g. "open" or "closed").

    Returns:
        Path to the complex PDB file for that ligand in that conformation.
    """
    return COMPLEX_DIR / f"FLE03_{name}_{CONFORMATIONS[conformation]}_model_0.pdb"


def select_ligand_conformation(name: str, asphericity: dict[str, float]) -> str:  # noqa: ARG001
    """Pick which conformation's pose becomes the single canonical ligand.

    Every ligand is modeled in both conformations, but it is charged once and
    the same charged molecule is reused for both protein conformations, so
    exactly one pose is chosen per ligand. The default policy keeps the more
    aspherical (more extended) pose. Asphericity is a shape descriptor,
    not a measure of pose accuracy or experimental resolution. Edit this to force a
    fixed conformation or an explicit per-ligand map; `name` is accepted (and
    ignored by the default policy) so per-ligand rules are easy to add.

    Args:
        name: Ligand name (accepted so explicit per-ligand policies are easy).
        asphericity: Mapping of conformation -> Asphericity of this ligand's
            pose in that conformation.

    Returns:
        The chosen conformation key.
    """
    return max(asphericity, key=asphericity.get)


def align_to_reference(
    mobile: mda.Universe,
    reference: mda.Universe,
    selection: str,
    *,
    cutoff: float = 2.0,
    max_cycles: int = 5,
) -> tuple[float, float, int, int]:
    """Superpose `mobile` onto `reference`, PyMOL-style.

    A single least-squares fit fails when two domains move relative to each
    other: it finds a global-minimum compromise that aligns neither well. This
    instead fits the `selection` atoms, then iteratively rejects atom pairs
    deviating more than `cutoff` and refits (up to `max_cycles`), so a rigid
    sub-domain (here the NTD) drives the alignment. The resulting rotation and
    translation are applied to the ENTIRE mobile universe, so the ligand and all
    other chains follow rigidly.

    Mobile and reference atoms are paired by (segid, resid, atom name), so the
    fit is correct even if the two selections enumerate atoms in a different
    order or come from heterogeneous inputs.

    Args:
        mobile: Universe to move (modified in place).
        reference: Universe to align onto.
        selection: MDAnalysis selection of atoms to fit (must match in both).
        cutoff: Per-atom deviation (A) above which a pair is rejected each cycle.
        max_cycles: Maximum number of outlier-rejection refinement cycles.

    Returns:
        (initial_rmsd, final_rmsd, n_kept, n_total) over the selection atoms.

    Raises:
        ValueError: If the selection is empty in either universe, or the two
            selections do not describe the same set of (segid, resid, name) atoms.
    """
    mob = mobile.select_atoms(selection)
    ref = reference.select_atoms(selection)
    if mob.n_atoms == 0 or ref.n_atoms == 0:
        raise ValueError(
            f"selection {selection!r} matched no atoms "
            f"(mobile={mob.n_atoms}, reference={ref.n_atoms})"
        )
    # Pair atoms by (segid, resid, name) rather than relying on enumeration
    # order, so the fit stays correct for heterogeneous or reordered inputs.
    ref_keys = list(zip(ref.segids, ref.resids, ref.names, strict=True))
    ref_index = {key: i for i, key in enumerate(ref_keys)}
    if len(ref_index) != ref.n_atoms:
        raise ValueError(
            f"reference selection {selection!r} has duplicate (segid, resid, name) atoms"
        )
    try:
        order = [ref_index[key] for key in zip(mob.segids, mob.resids, mob.names, strict=True)]
    except KeyError as exc:
        raise ValueError(
            f"mobile atom {exc.args[0]} has no (segid, resid, name) match in reference"
        ) from exc
    if len(order) != ref.n_atoms:
        raise ValueError(
            f"selection {selection!r} atom-count mismatch: "
            f"mobile={mob.n_atoms} vs reference={ref.n_atoms}"
        )
    ref_xyz = ref.positions[order].copy()
    initial = float(np.sqrt(((mob.positions - ref_xyz) ** 2).sum(1).mean()))
    keep = np.ones(mob.n_atoms, dtype=bool)
    for _ in range(max_cycles):
        m, r = mob.positions[keep], ref_xyz[keep]
        mc, rc = m.mean(0), r.mean(0)
        u, _, vt = np.linalg.svd((m - mc).T @ (r - rc))
        chirality = np.sign(np.linalg.det(vt.T @ u.T))
        rot = vt.T @ np.diag([1.0, 1.0, chirality]) @ u.T
        mobile.atoms.positions = (mobile.atoms.positions - mc) @ rot.T + rc
        new_keep = np.linalg.norm(mob.positions - ref_xyz, axis=1) < cutoff
        if np.array_equal(new_keep, keep) or new_keep.sum() < 3:
            if new_keep.sum() >= 3:
                keep = new_keep
            break
        keep = new_keep
    final = float(np.sqrt(((mob.positions[keep] - ref_xyz[keep]) ** 2).sum(1).mean()))
    return initial, final, int(keep.sum()), int(mob.n_atoms)


WORKING_DIR = Path("tmp/rbfe_open_closed")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

ALIGN_DIR = WORKING_DIR / "align"
ALIGN_DIR.mkdir(exist_ok=True)
LIGAND_DIR = WORKING_DIR / "ligands"
LIGAND_DIR.mkdir(exist_ok=True)
CHARGED_LIGAND_ROOT = WORKING_DIR / "charged_ligands"
NETWORK_DIR = WORKING_DIR / "networks"
NETWORK_DIR.mkdir(exist_ok=True)

# Per-conformation template proteins and OpenFE transformation outputs. The
# ligand directory, charges, and ligand network above are shared across both
# conformations so the comparison uses one identical set of charged ligands.
PROTEIN_PATHS = {conf: WORKING_DIR / f"protein_{conf}.pdb" for conf in CONFORMATIONS}
FIXED_PROTEIN_PATHS = {conf: WORKING_DIR / f"protein_{conf}_fixed.pdb" for conf in CONFORMATIONS}
# Simulation parameters
LAMBDA_WINDOWS = 51
N_REPLICAS = LAMBDA_WINDOWS
PH = 7.0
PRODUCTION_LENGTH = 20.0
# One repeat per job; quickrun.sh submits three independent jobs by default.
# N_REPLICAS counts interacting lambda replicas, not independent repeats.
REPEATS = 1
TEMPERATURE = 298.15

LOMAP_MAPPER_SEED = "O=C(CCCC[C,N])OP(OC[C@H]1O[C@@H](n2cnc3c2ncnc3N)[C@H](O)[C@@H]1O)([O-])=O"


@dataclass
class LomapAtomMapperConfig:
    """Configuration for the Lomap atom mapper."""

    time: int = 20
    threed: bool = True
    max3d: float = 1.0
    element_change: bool = False  # Do not allow mappings that change an atoms element
    seed: str = LOMAP_MAPPER_SEED  # SMARTS string for MCS search
    shift: bool = False


@dataclass
class KartografAtomMapperConfig:
    """Configuration for the Kartograf atom mapper."""

    atom_max_distance: float = 0.9
    atom_map_hydrogens: bool = True
    map_hydrogens_on_hydrogens_only: bool = True
    map_exact_ring_matches_only: bool = True
    allow_partial_fused_rings: bool = False
    allow_bond_breaks: bool = False


# Versions are part of cache identity; settings are recorded in the JSON protocols.
SOFTWARE_VERSIONS = {
    package: version(package)
    for package in ("openfe", "gufe", "openff-toolkit", "rdkit", "mdpp", "lomap2", "kartograf")
}

## Prepare structures from complex PDB files

Each complex PDB contains a protein and a ligand. For every ligand we compute the Asphericity of its pose in **both** conformations. This drives two choices: per conformation, the most aspherical ligand's complex becomes that conformation's **template protein**; per ligand, `select_ligand_conformation` picks the single **canonical pose** (default: the more aspherical conformation).

All complexes from both conformations are then superposed onto one common reference -- the `ALIGN_REFERENCE_CONFORMATION` template (open by default). Because the open and closed states differ by a large C-terminal-domain reorientation, a single least-squares fit (whole protein, or even all of the NTD) settles on a poor global-minimum compromise and misplaces the ligand. We therefore use `align_to_reference`, a PyMOL-style superposition that fits the NTD selection (`ALIGN_SELECTION`, residues 1-246) and iteratively rejects outlier atoms (`ALIGN_CUTOFF`/`ALIGN_MAX_CYCLES`), so the rigid NTD core drives the alignment (NTD RMSD ~0.5 A) while the moving CTD and the flexible ~166-182 loop are excluded automatically. The common frame lets us inspect the same ligand poses in both pockets; it does not guarantee that every pose remains appropriate in both states. Asphericity is only the deterministic pose/reference selection heuristic used here, not evidence of better structural resolution. Check contacts and ligand geometry, and compare alternative starting poses for flexible ligands.

We extract and protonate both template proteins with `fix_pdb`. The default model-pKa policy is a fixed-protonation approximation: identical pH and sequence do not guarantee identical histidine tautomers or even protonation states, because local hydrogen bonding can influence OpenMM's choices. Inspect the generated hydrogens, termini, and pocket residues (including His267) in both models. `thermo_settings.ph` does not make these constant-pH simulations or enumerate ligand protonation states. Ligand microstates come from the supplied SMILES. Resolve preparation warnings from the resulting structures rather than assuming they are benign. If the relevant physical protonation states differ, their free-energy/pH contributions need explicit treatment; forcing equal net charge is not a substitute.


In [ ]:
# Build SMILES template dict from .smi file
smi_suppl = Chem.SmilesMolSupplier(str(SMILES_PATH), sanitize=True)
smiles_dict = {mol.GetProp("_Name"): mol for mol in smi_suppl if mol is not None}

# Pass 1: for every ligand, extract its pose in BOTH conformations and compute
# Asphericity. Two things are decided here:
#   - the template protein per conformation = complex bound to the most
#     aspherical ligand in that conformation;
#   - the single canonical pose per ligand, chosen by select_ligand_conformation
#     (this pose is charged once and reused for both protein conformations).
ligand_asphericity = {conf: {} for conf in CONFORMATIONS}
ligand_volume = {conf: {} for conf in CONFORMATIONS}
for conf in CONFORMATIONS:
    for lig_name, template in smiles_dict.items():
        pdb_file = complex_pdb_for(lig_name, conf)
        ligand_pdb = ALIGN_DIR / f"{lig_name}_{conf}.pdb"
        u = mda.Universe(pdb_file)
        u.select_atoms(f"chainID {LIGAND_CHAIN_ID}").write(ligand_pdb)
        rdkit_mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
        rdkit_mol = assign_topology(rdkit_mol, template)
        ligand_asphericity[conf][lig_name] = Asphericity(rdkit_mol)
        ligand_volume[conf][lig_name] = AllChem.ComputeMolVolume(rdkit_mol)

# Template protein per conformation = bound to the most aspherical ligand
template_name = {
    conf: max(ligand_asphericity[conf], key=ligand_asphericity[conf].get) for conf in CONFORMATIONS
}
# Single global alignment frame (e.g. the open template protein)
ref_pdb = complex_pdb_for(template_name[ALIGN_REFERENCE_CONFORMATION], ALIGN_REFERENCE_CONFORMATION)
# Canonical pose conformation per ligand (used for charges and ligand network)
ligand_conf = {
    lig_name: select_ligand_conformation(
        lig_name, {conf: ligand_asphericity[conf][lig_name] for conf in CONFORMATIONS}
    )
    for lig_name in smiles_dict
}

for conf in CONFORMATIONS:
    print(f"[{conf}] template protein: {template_name[conf]}")
    print(f"{'Ligand':<40} {'Asphericity':>12} {'Volume (A^3)':>12}")
    for lig_name in sorted(ligand_asphericity[conf]):
        marker = " <-- template" if lig_name == template_name[conf] else ""
        marker += " *canonical pose*" if ligand_conf[lig_name] == conf else ""
        print(
            f"{lig_name:<40} {ligand_asphericity[conf][lig_name]:>12.4f}"
            f" {ligand_volume[conf][lig_name]:>12.2f}{marker}"
        )
    print()

In [ ]:
# Pass 2: superpose every complex (both conformations) onto the common
# reference frame, then extract (a) the canonical ligand pose from its chosen
# conformation and (b) each conformation's template protein -- all in the
# shared frame so the single ligand set sits in both pockets.
ref_u = mda.Universe(ref_pdb)

for conf in CONFORMATIONS:
    for lig_name, template in smiles_dict.items():
        pdb_file = complex_pdb_for(lig_name, conf)
        mob_u = mda.Universe(pdb_file)
        init_rmsd, final_rmsd, n_kept, n_total = align_to_reference(
            mob_u, ref_u, ALIGN_SELECTION, cutoff=ALIGN_CUTOFF, max_cycles=ALIGN_MAX_CYCLES
        )
        # Extract the canonical ligand only from its selected conformation
        if ligand_conf[lig_name] == conf:
            ligand_pdb = ALIGN_DIR / f"{lig_name}_aligned.pdb"
            mob_u.select_atoms(f"chainID {LIGAND_CHAIN_ID}").write(ligand_pdb)
            rdkit_mol = Chem.MolFromPDBFile(str(ligand_pdb), removeHs=True)
            rdkit_mol = assign_topology(rdkit_mol, template)
            rdkit_mol.SetProp("_Name", lig_name)
            Chem.MolToMolFile(rdkit_mol, str(LIGAND_DIR / f"{lig_name}.sdf"))
            print(
                f"{lig_name} [{conf}]  (NTD RMSD {init_rmsd:.2f} -> {final_rmsd:.2f} A"
                f" on {n_kept}/{n_total} atoms)"
            )
            view_mol_3d(rdkit_mol, width=400, height=300, show=True)
        # Extract this conformation's template protein
        if lig_name == template_name[conf]:
            mob_u.select_atoms(f"chainID {PROTEIN_CHAIN_ID}").write(PROTEIN_PATHS[conf])

In [ ]:
protein_trajs = {}
for conf in CONFORMATIONS:
    fix_pdb(PROTEIN_PATHS[conf], FIXED_PROTEIN_PATHS[conf], PH)
    protein_trajs[conf] = md.load_pdb(str(FIXED_PROTEIN_PATHS[conf]))
    print(f"[{conf}] template protein: {template_name[conf]}")
    display(view_traj_3d(protein_trajs[conf]))

## Loading the ligands

We load the pre-aligned ligand SDF files produced in the previous step. Each file has correct bond orders and hydrogens assigned via `assign_topology`.

In [ ]:
ligands_rdmol = sorted(
    [
        Chem.MolFromMolFile(str(f), removeHs=False)
        for f in [LIGAND_DIR / f"{name}.sdf" for name in sorted(smiles_dict)]
    ],
    key=lambda m: m.GetProp("_Name"),
)
ligands = [SmallMoleculeComponent(mol, name=mol.GetProp("_Name")) for mol in ligands_rdmol]

ligand_labels = [mol.GetProp("_Name") if mol.HasProp("_Name") else "" for mol in ligands_rdmol]
seed_pattern = Chem.MolFromSmarts(LOMAP_MAPPER_SEED) if LOMAP_MAPPER_SEED.strip() else None

draw_mols(
    ligands_rdmol,
    legends=ligand_labels,
    pattern=seed_pattern,
    mols_per_row=4,
    sub_img_size=(300, 300),
)

## Charging the ligands

It is recommended to use a single set of charges for each ligand to ensure reproducibility between repeats or consistent charges between different legs of a calculation involving the same ligand, like a relative binding affinity calculation for example. 

Here we will use some utility functions from OpenFE which can assign partial charges to a series of molecules with a variety of methods which can be configured via the `OpenFFPartialChargeSettings` class. In this example 
we will charge the ligands using the `am1bcc` method from `ambertools` which is the default charge scheme used by OpenFE.

> **Warning:** AM1-BCC charge generation is slow (can take several minutes per ligand). To avoid recomputing charges on every run, the charged molecules are saved as SDF files to `CHARGED_LIGAND_DIR`. The cache is keyed by the exact current ligand SDFs, charge settings, and software versions. A changed pose, chemical identity, or charge method creates a new cache; partial caches are regenerated.

In [ ]:
charge_settings = OpenFFPartialChargeSettings(
    partial_charge_method="am1bcc", off_toolkit_backend="ambertools"
)
charge_key = input_digest(
    [LIGAND_DIR / f"{name}.sdf" for name in sorted(smiles_dict)],
    settings={"charges": charge_settings.model_dump(mode="json"), "versions": SOFTWARE_VERSIONS},
)
CHARGED_LIGAND_DIR = CHARGED_LIGAND_ROOT / charge_key
CHARGED_LIGAND_DIR.mkdir(parents=True, exist_ok=True)
charged_sdfs = [CHARGED_LIGAND_DIR / f"{lig.name}.sdf" for lig in ligands]
if all(path.exists() for path in charged_sdfs):
    charged_ligands = [SmallMoleculeComponent.from_sdf_file(str(f)) for f in charged_sdfs]
else:
    charged_ligands = bulk_assign_partial_charges(
        molecules=ligands,
        overwrite=True,
        method=charge_settings.partial_charge_method,
        toolkit_backend=charge_settings.off_toolkit_backend,
        generate_n_conformers=charge_settings.number_of_conformers,
        nagl_model=charge_settings.nagl_model,
        processors=mp.cpu_count(),
    )
    for smc in charged_ligands:
        (CHARGED_LIGAND_DIR / f"{smc.name}.sdf").write_text(smc.to_sdf())

In [ ]:
for lig in charged_ligands:
    rdmol = lig.to_rdkit()
    labels = make_atom_labels_3d(
        rdmol,
        text_fn=lambda atom: f"{atom.GetDoubleProp('PartialCharge'):+.2f}",
        color_fn=lambda atom: "red" if atom.GetDoubleProp("PartialCharge") < 0 else "blue",
        base_style={
            "fontSize": 10,
            "showBackground": True,
            "backgroundOpacity": 0.6,
            "backgroundColor": "white",
        },
    )
    print(lig.name)
    view_mol_3d(
        rdmol,
        width=800,
        height=400,
        style={"stick": {}, "sphere": {"scale": 0.3}},
        labels=labels,
        show=True,
    )

## Creating the `LigandNetwork`

The first step is to create a `LigandNetwork`, which is a network with small molecules as nodes, and atom mappings, the description of how to alchemically mutate between the molecules, as its edges.

The pipeline for creating a `LigandNetwork` can involve three components:

* **Atom Mapper**: Proposes potential atom mappings (descriptions of the alchemical change) for pairs of ligands.
* **Scorer**: Given an atom mapping, provides an estimate of the quality of that mapping (higher scores are better).
* **Network Planner**: Creates the actual `LigandNetwork`; different network planners provide different strategies.

Each of these components could be replaced by other options.

In [ ]:
lomap_mapper = LomapAtomMapper(**asdict(LomapAtomMapperConfig()))
kartograf_mapper = KartografAtomMapper(**asdict(KartografAtomMapperConfig()))

print(LomapAtomMapperConfig())
print(KartografAtomMapperConfig())

# Passing multiple atom mappers to the planner
mappers = [lomap_mapper, kartograf_mapper]
scorer = default_lomap_score
network_planner = generate_minimal_redundant_network

The exact call signature depends on the network planner: a minimal spanning network requires a score, whereas that is optional for a radial network (but a radial network needs the central ligand to be provided).

If a previously saved `graphml` network exists on disk, it is loaded directly; otherwise a new network is planned and written out for reuse:

Now we can look at the overall structure of the `LigandNetwork`:

In [ ]:
network_key = input_digest(
    charged_sdfs,
    settings={
        # Serialize the active mapper instances in their actual execution order.
        "mappers": [mapper.to_json() for mapper in mappers],
        "scorer": f"{scorer.__module__}.{scorer.__name__}",
        "planner": f"{network_planner.__module__}.{network_planner.__name__}",
        "versions": SOFTWARE_VERSIONS,
    },
)
LIGAND_NETWORK_PATH = NETWORK_DIR / f"{network_key}.graphml"
if LIGAND_NETWORK_PATH.exists():
    graphml_str = LIGAND_NETWORK_PATH.read_text(encoding="utf-8")
    ligand_network = openfe.LigandNetwork.from_graphml(graphml_str)
else:
    ligand_network = network_planner(ligands=charged_ligands, mappers=mappers, scorer=scorer)
    with LIGAND_NETWORK_PATH.open("w", encoding="utf-8") as f:
        f.write(ligand_network.to_graphml())

_ = plot_atommapping_network(ligand_network)

We can also inspect the individual atom mappings:

To get the score for this mapping, we inspect its `annotations` attribute. Arbitrary annotations can be added when a mapping is created, although our network generator only includes the score.

In [ ]:
for mapping in ligand_network.edges:
    print(mapping.annotations)
    display(mapping)
    display(mapping.view_3d(show_atomIDs=True))

## Creating `Transformation`s

The `LigandNetwork` only knows about the small molecules and the alchemical connections between them. It doesn't know anything about environment (e.g., solvent) or about the `Protocol` that will be used during the simulation.

That information is included in a `Transformation`. Each edge in the `LigandNetwork` becomes two `Transformation`s: one complex leg and one solvent leg. We build the shared components -- a `SolventComponent`, a `ProteinComponent` **per conformation**, and two `Protocol`s (one per leg) -- and then assemble **one `AlchemicalNetwork` per conformation**. The ligands, mappings, and charges are identical across the two networks; only the complex-leg protein changes.

### Creating `ChemicalSystem`s

OpenFE describes complex molecular systems as being composed of `Component`s. For example, we have `SmallMoleculeComponent` for each small molecule in the `LigandNetwork`. We'll create a `SolventComponent` to describe the solvent, and binding free energy calculations involve a `ProteinComponent`.

The `Component`s are joined in a `ChemicalSystem`, which describes all the particles in the simulation.

In [ ]:
# defaults are water with NaCl at 0.15 M
solvent = SolventComponent()

In [ ]:
proteins = {
    conf: ProteinComponent.from_pdb_file(
        str(FIXED_PROTEIN_PATHS[conf]), name=FIXED_PROTEIN_PATHS[conf].stem
    )
    for conf in CONFORMATIONS
}

### Creating a `Protocol`

The actual simulation is performed by a `Protocol`. We'll use an OpenMM-based hybrid topology relative free energy `Protocol`.

The easiest way to customize protocol settings is to start with the default settings, and modify them. Many settings carry units with them.

We use the default settings with an adapted solvent padding for the complex phase to avoid adding too many waters.

**Note: Since [OpenFE v1.5.0](https://docs.openfree.energy/en/v1.6.1/CHANGELOG.html#v1-5-0), `real_time_analysis_interval` does not need to be divisible by `checkpoint_interval`.**

**Note: by default the settings use a solvent padding of 1.5 nm, this is appropriate for solvent simulations, but for complex simulations this leads to excess water being in the box, slowing down your simulation. To deal with this, we will create two sets of settings, and two sets of protocols, one for each leg of the transformations.**

In [ ]:
complex_settings = RelativeHybridTopologyProtocol.default_settings()

# Change the value
complex_settings.protocol_repeats = REPEATS
complex_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
complex_settings.thermo_settings.ph = PH
complex_settings.simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond
complex_settings.simulation_settings.n_replicas = N_REPLICAS
complex_settings.lambda_settings.lambda_windows = LAMBDA_WINDOWS

# Reduce the solvent padding for the complex phase
complex_settings.solvation_settings.solvent_padding = 1.0 * unit.nanometer

complex_protocol = RelativeHybridTopologyProtocol(complex_settings)

complex_settings

In [ ]:
solvent_settings = RelativeHybridTopologyProtocol.default_settings()

# Change the value
solvent_settings.protocol_repeats = REPEATS
solvent_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
solvent_settings.thermo_settings.ph = PH
solvent_settings.simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond
solvent_settings.simulation_settings.n_replicas = N_REPLICAS
solvent_settings.lambda_settings.lambda_windows = LAMBDA_WINDOWS

# Keep the default solvent padding
solvent_settings.solvation_settings.solvent_padding = 1.5 * unit.nanometer

solvent_protocol = RelativeHybridTopologyProtocol(solvent_settings)

solvent_settings

### Creating the `Transformation`

Once we have the mapping, the two `ChemicalSystem`s, and the `Protocol`, creating the `Transformation` is easy:

Each `Transformation` contains:
- chemical models of both sides of the alchemical transformation in `stateA` and `stateB`
- the correspondence of atoms between the two sides in `mapping`
- a description of the exact computational algorithm to use (`complex_protocol` for the complex leg, `solvent_protocol` for the solvent leg)

### Building the `Transformation`s

With the mapping, the `ChemicalSystem`s, and the `Protocol`s in place, we loop over every edge in the `LigandNetwork` and create two `Transformation`s per edge (one complex leg, one solvent leg):

Alternatively, `RBFEAlchemicalNetworkPlanner` can build an `AlchemicalNetwork` in a single call. Here we construct it manually to show each step explicitly.

In [ ]:
# One AlchemicalNetwork per protein conformation. The ligand network, mappings,
# and charges are shared; only the complex-leg protein differs between them.
# (The solvent legs are conformation-independent and therefore identical across
# the two networks; they are kept in each so every network runs standalone.)
networks = {}
for conf in CONFORMATIONS:
    protein = proteins[conf]
    transformations = []
    for mapping in ligand_network.edges:
        for leg in ["solvent", "complex"]:
            # use the solvent and protein created above
            sysA_dict = {"ligand": mapping.componentA, "solvent": solvent}
            sysB_dict = {"ligand": mapping.componentB, "solvent": solvent}

            if leg == "complex":
                # If this is a complex transformation we use the complex protocol
                # and add in the protein to the chemical states
                protocol = complex_protocol
                sysA_dict["protein"] = protein
                sysB_dict["protein"] = protein
            else:
                # If this is a solvent transformation we just use the solvent protocol
                protocol = solvent_protocol

            # we don't have to name objects, but it can make things (like filenames) more convenient
            sysA = ChemicalSystem(sysA_dict, name=f"{mapping.componentA.name}_{leg}")
            sysB = ChemicalSystem(sysB_dict, name=f"{mapping.componentB.name}_{leg}")

            prefix = "rbfe_"  # prefix is only to exactly reproduce CLI

            transformation = Transformation(
                stateA=sysA,
                stateB=sysB,
                mapping=mapping,
                protocol=protocol,  # use protocol created above
                name=f"{prefix}{sysA.name}_{sysB.name}",
            )
            transformations.append(transformation)

    networks[conf] = AlchemicalNetwork(transformations)

## Writing the `AlchemicalNetwork` to disk

We'll write out each transformation to disk, so that they can be run independently using the `openfe quickrun` command:

In [ ]:
# Each set of inputs/settings has its own campaign directory, including results.
# This avoids old transformation JSONs or resumed jobs contaminating a new setup.
campaign_key = input_digest(
    [LIGAND_NETWORK_PATH, *FIXED_PROTEIN_PATHS.values()],
    settings={
        "complex": complex_settings.model_dump(mode="json"),
        "solvent": solvent_settings.model_dump(mode="json"),
        "versions": SOFTWARE_VERSIONS,
        "solvent_component": solvent.to_json(),
    },
)
CAMPAIGN_DIR = WORKING_DIR / "campaigns" / campaign_key
TRANSFORMATION_DIRS = {conf: CAMPAIGN_DIR / conf / "transformations" for conf in CONFORMATIONS}
for directory in TRANSFORMATION_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# Write out each transformation, one directory per conformation
for conf in CONFORMATIONS:
    for transformation in networks[conf].edges:
        transformation.to_json(TRANSFORMATION_DIRS[conf] / f"{transformation.name}.json")
    print(f"[{conf}] {len(os.listdir(TRANSFORMATION_DIRS[conf]))} transformations")

# List contents of each conformation's transformations directory
{conf: os.listdir(TRANSFORMATION_DIRS[conf]) for conf in CONFORMATIONS}
print(
    "Run quickrun.sh from:", [str(path.parent.resolve()) for path in TRANSFORMATION_DIRS.values()]
)

Each transformation JSON contains one hybrid-topology leg. Run `scripts/openfe/quickrun/quickrun.sh` from each printed campaign directory containing `transformations/`. Its default is three independent array jobs; these notebooks set `protocol_repeats=1` per job, so the default total is three repeats per leg. For standalone `openfe quickrun`, either invoke three jobs with distinct work/output paths or set `REPEATS=3` when generating the protocols. A restart/resume is a continuation of its original repeat, not new independent data.

For an A → B edge, combine hybrid-topology results as
`ΔΔG_bind = ΔG_complex(A→B) - ΔG_solvent(A→B)` in a common unit; negative means B binds more favorably than A. A single-repeat MBAR uncertainty describes within-run precision, while independent repeats probe between-run variation. Do not count lambda windows or repeated checkpoints as independent replicates. Keep the full result JSONs and gather all independent repeats with OpenFE.

Before interpreting affinities, inspect MBAR overlap, replica mixing, forward/reverse convergence, ligand pose/COM stability, and network cycle closure. Twenty ns and 51 lambda windows are starting settings, not evidence of convergence. More lambda windows cannot by themselves solve slow receptor or ligand transitions. See the [OpenFE analysis guide](https://docs.openfree.energy/en/latest/guide/protocols/relativehybridtopology.html) and [Klimovich et al. (2015)](https://pmc.ncbi.nlm.nih.gov/articles/PMC4420631/).


## SepTop Relative Binding Free Energy Transformations

SepTop runs separate complex and solvent simulations inside one protocol DAG, so export one transformation per ligand pair. Its final `get_estimate()` already returns the binding **ΔΔG**: do not subtract a separate solvent estimate from it again. Atom mappings are unnecessary (`mapping=None`). OpenFE applies and accounts for ligand orientational restraints; these do **not** constrain the receptor to an open or closed basin. Inspect restraint geometry, each leg's convergence, and between-repeat variation. See the [official SepTop guide](https://docs.openfree.energy/en/latest/guide/protocols/septop.html).


In [ ]:
septop_settings = SepTopProtocol.default_settings()

septop_settings.protocol_repeats = REPEATS
septop_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
septop_settings.thermo_settings.ph = PH

# lambda windows are handled by the septop protocol internally

septop_settings.complex_simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond
septop_settings.solvent_simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond

septop_protocol = SepTopProtocol(septop_settings)

septop_settings

In [ ]:
septop_key = input_digest(
    [LIGAND_NETWORK_PATH, *FIXED_PROTEIN_PATHS.values()],
    settings={
        "septop": septop_settings.model_dump(mode="json"),
        "versions": SOFTWARE_VERSIONS,
        "solvent_component": solvent.to_json(),
    },
)
SEPTOP_TRANSFORMATION_DIRS = {
    conf: WORKING_DIR / "septop_campaigns" / septop_key / conf / "transformations"
    for conf in CONFORMATIONS
}
for directory in SEPTOP_TRANSFORMATION_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# One SepTop AlchemicalNetwork per protein conformation, written to its own dir
septop_networks = {}
for conf in CONFORMATIONS:
    protein = proteins[conf]
    septop_transformations = []
    for mapping in ligand_network.edges:
        ligA, ligB = mapping.componentA, mapping.componentB
        stateA = ChemicalSystem(
            {"ligand": ligA, "protein": protein, "solvent": solvent},
            name=f"{ligA.name}_complex",
        )
        stateB = ChemicalSystem(
            {"ligand": ligB, "protein": protein, "solvent": solvent},
            name=f"{ligB.name}_complex",
        )

        prefix = "rbfe_"  # prefix is only to exactly reproduce CLI

        septop_transformations.append(
            Transformation(
                stateA=stateA,
                stateB=stateB,
                mapping=None,
                protocol=septop_protocol,
                name=f"{prefix}{ligA.name}_{ligB.name}_septop",
            )
        )

    septop_networks[conf] = AlchemicalNetwork(septop_transformations)

    for septop_transformation in septop_networks[conf].edges:
        septop_transformation.to_json(
            SEPTOP_TRANSFORMATION_DIRS[conf] / f"{septop_transformation.name}.json"
        )
    print(f"[{conf}] {len(os.listdir(SEPTOP_TRANSFORMATION_DIRS[conf]))} septop transformations")

{conf: os.listdir(SEPTOP_TRANSFORMATION_DIRS[conf]) for conf in CONFORMATIONS}